# Benchmark three Foundation Model API LLMs and three embedding models against the same prompt set, then pick the winners for a stated cost and latency budget

The chat-assistant lead has been on the platform for six months and is annoyed by "we should benchmark first" meetings that never produce numbers. You have one session to produce the numbers: three LLMs on the same five prompts, three embedding models on the same three queries, two paragraphs of justification, one Delta table the team can SELECT against next quarter when they want to revisit the choice. No more "we should benchmark first." This is the benchmark.

The constraint scenario: an internal helpdesk doing 100 queries per day under a 200 ms p95 latency target on first token, a 0.50 USD per day token budget, source documents chunked at 512 tokens. English only, requests 50 to 200 tokens, responses 100 to 400 tokens. Pick one LLM and one embedding model that fit.

The hands-on work:

- Discover Foundation Model API endpoints visible from your Free Edition workspace and split them into chat-completion LLMs and embedding endpoints.
- Run the same five-prompt benchmark through three chat LLMs, capturing latency, prompt tokens, completion tokens, and cost-per-call.
- Run the same three reference questions through three embedding endpoints, capturing dimension, latency, and cost.
- Write two INSERT statements that persist your chosen LLM and chosen embedding model plus a justification paragraph that names the constraint each choice satisfies.

**Time.** Plan for about 55 minutes hands-on. First-call cold-start adds 5 to 20 seconds per endpoint; the benchmark loop discards the warm-up call and reports timing on the rest. Budget up to 90 minutes for the session window.

**Cost.** About 1 to 2 cents on Databricks-hosted endpoints (25 endpoint calls, blended pay-per-token pricing). If you wire up an external OpenAI endpoint via external_models, expect another 1 to 5 cents against your OpenAI account; the lab has a skip-this-branch path that uses a third Databricks-hosted endpoint instead. Session range: $0.01 to $0.10. Your morning coffee cost more.

This lab maps to Databricks GenAI Engineer Associate Domain 1 (Design Applications) and Domain 3 (Application Development). Prerequisite: Lab 1 (`prompt-engineering-and-foundation-model-api-basics`) completed.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks, the Databricks SDK, and the OpenAI Python client.
# Pinned versions per LAB-CREATION-BLUEPRINT.md build rules. The Foundation
# Model API speaks the OpenAI chat-completions and embeddings shape, so the
# stock openai client works against /serving-endpoints with a Databricks PAT.

!pip install --quiet labrun-checks==0.7.0 databricks-sdk==0.40.0 openai==1.54.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Three constant blocks:
# 1. UC identifiers (catalog, schema, table FQNs). snake_case under labrun_.
# 2. Benchmark fixtures (5 prompts of varying length and complexity, 3
#    reference queries for the embedding loop).
# 3. PRICING dict with per-1k-token rates. Verified against the Databricks
#    Foundation Model API pricing page on PRICING_VERIFIED_ON; the wallet
#    helper warns if that date is older than 90 days.

import atexit
import getpass
import sys
import time
from datetime import date, datetime
from time import perf_counter_ns

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState
from openai import OpenAI

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "select-llm-and-embedding-model-from-foundation-model-api"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[1].slug exactly

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_model_selection"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"

BENCHMARK_PROMPTS_TABLE_FQN = f"{LAB_SCHEMA_FQN}.benchmark_prompts"
LLM_RESULTS_TABLE_FQN = f"{LAB_SCHEMA_FQN}.llm_benchmark_results"
EMBEDDING_RESULTS_TABLE_FQN = f"{LAB_SCHEMA_FQN}.embedding_benchmark_results"
DECISIONS_TABLE_FQN = f"{LAB_SCHEMA_FQN}.decisions"

# Five prompts, intentionally varied. The benchmark loop runs each against
# each of three LLM endpoints. A 15-row result table is what Checkpoint 2
# asserts.
BENCHMARK_PROMPTS = [
    {
        "id": 1,
        "name": "short_factual",
        "system": "You answer in one sentence. No preamble.",
        "user": "What is the capital of Japan?",
    },
    {
        "id": 2,
        "name": "medium_summary",
        "system": "You summarize text in three bullet points. Return only the bullets.",
        "user": (
            "Summarize: Mosaic AI Vector Search is a vector database integrated "
            "with Databricks. It supports Delta-Sync indexes that keep an index "
            "automatically aligned with a source Delta table, and Direct Vector "
            "Access indexes where the engineer manages the embedding lifecycle "
            "directly. Free Edition supports Delta-Sync only."
        ),
    },
    {
        "id": 3,
        "name": "long_reasoning",
        "system": "You reason step by step before answering. Show the steps.",
        "user": (
            "An internal helpdesk handles 100 queries per day. Each query costs "
            "0.003 USD on a 70B model and 0.0005 USD on a 7B model. The latency "
            "target is 200 ms p95. The 70B endpoint averages 600 ms first token "
            "under typical load. The 7B endpoint averages 150 ms. The daily "
            "budget is 0.50 USD. Which model fits both constraints?"
        ),
    },
    {
        "id": 4,
        "name": "json_format",
        "system": (
            "You return a single JSON object with keys 'intent' (string), "
            "'priority' (low|medium|high), and 'customer_id' (string). No prose."
        ),
        "user": (
            "Customer email: 'My order #ORD-7741 has not arrived in three weeks "
            "and the tracking link is dead. I need this resolved today.' Extract."
        ),
    },
    {
        "id": 5,
        "name": "classification",
        "system": "You classify the sentiment as positive, neutral, or negative. One word reply.",
        "user": (
            "Review: 'Setup took ten minutes, the docs were clear, and the first "
            "query came back in under a second. I am keeping this on the team.' "
            "Classify."
        ),
    },
]

# Three reference queries for the embedding benchmark. 3 endpoints x 3
# queries = 9 rows in the embedding result table (Checkpoint 3).
EMBEDDING_QUERIES = [
    "How do I sign up for Databricks Free Edition?",
    "What is the maximum context length on databricks-gte-large-en?",
    "Can a Delta-Sync index keep itself aligned with a source Delta table automatically?",
]

# Foundation Model API pay-per-token rates. Verified on PRICING_VERIFIED_ON
# against the Databricks pricing page that accompanies the March 18 2026
# exam guide. The setup cell warns when this stamp is older than 90 days.
# Rates are USD per 1,000 tokens (input and output split where the endpoint
# bills asymmetrically; embedding endpoints bill input-only).
PRICING_VERIFIED_ON = "2026-03-18"

PRICING = {
    "databricks-meta-llama-3-3-70b-instruct": {
        "kind": "chat",
        "input_per_1k": 0.00100,
        "output_per_1k": 0.00300,
    },
    "databricks-mixtral-8x7b-instruct": {
        # Documented small-model substitute. If the workspace has renamed
        # this endpoint, the discovery step in Task 1 falls back to whatever
        # the current small-model endpoint name is and prices it at this
        # rate (the rate is what the exam tests; the name is a moving
        # target).
        "kind": "chat",
        "input_per_1k": 0.00050,
        "output_per_1k": 0.00100,
    },
    "databricks-llama-3-1-8b-instruct": {
        # First fallback if mixtral has been retired in the current workspace.
        "kind": "chat",
        "input_per_1k": 0.00020,
        "output_per_1k": 0.00060,
    },
    "databricks-gte-large-en": {
        "kind": "embedding",
        "input_per_1k": 0.00010,
        "output_per_1k": 0.00000,
    },
    "databricks-bge-large-en": {
        "kind": "embedding",
        "input_per_1k": 0.00010,
        "output_per_1k": 0.00000,
    },
    "databricks-bge-small-en": {
        "kind": "embedding",
        "input_per_1k": 0.00002,
        "output_per_1k": 0.00000,
    },
}

# Expected dimensions for embedding endpoints. Used by Checkpoint 3 as a
# warn-only lookup; a mismatch is logged, not failed.
EXPECTED_EMBEDDING_DIMENSIONS = {
    "databricks-gte-large-en": 1024,
    "databricks-bge-large-en": 1024,
    "databricks-bge-small-en": 384,
}

# Constraint scenario for the decision row. The Task 4 justification text
# must reference at least one of these keywords (latency, cost, token,
# recall, dimension, context) per Checkpoint 4.
CONSTRAINT_SCENARIO = (
    "100 queries per day, 200 ms p95 latency on first token, 0.50 USD per day "
    "token budget, source chunks at 512 tokens, English only."
)

In [ ]:
# NBVAL_SKIP
# Validate Databricks credentials via the SDK, resolve a SQL warehouse so the
# SQL Statement Execution API works for table writes and reads, build the
# OpenAI-compatible client pointed at /serving-endpoints, and register the
# session with labrun-checks. This cell must succeed before the manifest cell
# runs anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL (https://...azuredatabricks.net or .cloud.databricks.com): ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"
# Trim trailing slash because both /serving-endpoints and the SDK expect a
# bare host.
databricks_host = databricks_host.rstrip("/")

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    print()
    print("Refresh your workspace URL and PAT, then re-run this cell.")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found in this workspace. Free Edition ships with")
    print("a Starter warehouse by default; if it has been deleted, recreate it")
    print("from the SQL > Warehouses page before re-running this cell.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    """Submit a SQL statement to the warehouse and return the parsed result.

    Polls statement state up to wait_seconds, raises on FAILED or CANCELED,
    returns a dict {columns: [...], rows: [[...]]} on SUCCEEDED. Identical
    contract to the helper in Lab 1 so muscle memory carries forward.
    """
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid,
        statement=statement,
        wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}. "
                f"The Starter warehouse may still be waking up; re-run the cell."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)

    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


# OpenAI-compatible client pointed at the workspace's serving-endpoints
# surface. The Foundation Model API speaks the chat.completions and
# embeddings shape; the model parameter is the endpoint name (e.g.,
# "databricks-meta-llama-3-3-70b-instruct").
openai_client = OpenAI(
    api_key=databricks_token,
    base_url=f"{databricks_host}/serving-endpoints",
)

# Pricing freshness check. Warn if the published rates are older than 90
# days because pay-per-token pricing drifts and the lab's cost math relies
# on verified numbers.
try:
    _verified = datetime.strptime(PRICING_VERIFIED_ON, "%Y-%m-%d").date()
    _age_days = (date.today() - _verified).days
    if _age_days > 90:
        print(
            f"PRICING WARNING: PRICING_VERIFIED_ON={PRICING_VERIFIED_ON} is "
            f"{_age_days} days old. Re-verify the per-1k-token rates against "
            f"the current Databricks Foundation Model API pricing page before "
            f"quoting the cost numbers from this lab."
        )
    else:
        print(f"Pricing dict verified {_age_days} days ago. Cost math is current.")
except Exception:
    pass

register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest is module-level and in reverse-creation order (decisions first,
# then the two result tables, then the prompts table, then the schema).
# Free Edition has no hourly-billed resources for this lab; the only thing
# the lab spends is pay-per-token Foundation Model API calls, which bill
# at request time and are not affected by cleanup. Per RESOURCE-SAFETY-SPEC
# Rule 4 the orphan scan blocks execution if any tagged resources from a
# prior session exist.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_managed_table",
        id=DECISIONS_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {DECISIONS_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=EMBEDDING_RESULTS_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {EMBEDDING_RESULTS_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=LLM_RESULTS_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {LLM_RESULTS_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=BENCHMARK_PROMPTS_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {BENCHMARK_PROMPTS_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    """Inline adapter implementing the labrun-checks CleanupAdapter protocol
    against Databricks Unity Catalog via the SQL Statement Execution API.
    Lab 2 only touches uc_managed_table and uc_schema; the wider type list
    from the DE Associate template is preserved so a future lab can extend
    without rewriting the adapter shape."""

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        else:
            raise ValueError(f"DatabricksCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        """Return list of Unity Catalog identifiers tagged with the lab slug.

        Reads schema_tags and table_tags; the orphan scan compares results
        against the manifest to surface leftovers from a previous session."""
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate rows in the")
    print("benchmark result tables and break the checkpoint row-count asserts.")
    print("Run the cleanup cell at the bottom of this notebook first, or drop")
    print("them manually with the SQL commands shown by the cleanup cell, then")
    print("re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Create the lab schema, create four Delta tables, and seed the benchmark prompts

Five SQL statements and a serving-endpoint discovery step.

1. `CREATE SCHEMA IF NOT EXISTS workspace.default.labrun_model_selection` and tag it with the lab slug.
2. Create four Delta tables under the schema:
   - `benchmark_prompts (prompt_id INT, name STRING, system_message STRING, user_message STRING)`
   - `llm_benchmark_results (endpoint_name STRING, endpoint_type STRING, prompt_id INT, latency_ms DOUBLE, prompt_tokens INT, completion_tokens INT, cost_usd_estimate DOUBLE, response_excerpt STRING)`
   - `embedding_benchmark_results (endpoint_name STRING, endpoint_type STRING, query_id INT, query_text STRING, embedding_dimension INT, latency_ms DOUBLE, cost_usd_estimate DOUBLE)`
   - `decisions (decision_type STRING, chosen_endpoint STRING, constraint_summary STRING, justification_text STRING)`
   Tag each table with the lab slug.
3. Insert the five rows from `BENCHMARK_PROMPTS` into `benchmark_prompts`.
4. Discover Foundation Model API endpoints via `WorkspaceClient.serving_endpoints.list()`. Filter to chat-completion LLMs and to embedding endpoints. Pick three of each. The lab prefers `databricks-meta-llama-3-3-70b-instruct` for the large chat slot, `databricks-mixtral-8x7b-instruct` (or the current small-model substitute) for the small chat slot, and either an external_models GPT-class endpoint or a third Databricks-hosted chat endpoint for the third slot. For embeddings: `databricks-gte-large-en`, `databricks-bge-large-en`, and `databricks-bge-small-en`.

The discovery step writes nothing to the result tables yet (Checkpoint 1 reads from the LLM and embedding result tables, which Task 2 and Task 3 populate). The schema, the four tables, the prompt seed, and the resolved endpoint name lists are all this task produces.

If your workspace has rotated the small-model endpoint name (Databricks rotates these quarterly), the fallback chain in Task 2 names the alternatives. The exam tests "match the endpoint to the constraint," not "memorize the endpoint name."

> Tip: `endpoint.task` on a serving endpoint reads `"llm/v1/chat"` for chat completions and `"llm/v1/embeddings"` for embeddings. Use this to split the catalog.

In [ ]:
# NBVAL_SKIP
# Task 1: schema + four tables + tag application + prompt seed + endpoint
# discovery. The discovery output (LLM_ENDPOINTS, EMBEDDING_ENDPOINTS) feeds
# Tasks 2 and 3.

CHAT_TASK_LABELS = {"llm/v1/chat", "llm/v1/completions"}
EMBEDDING_TASK_LABELS = {"llm/v1/embeddings"}

# Preferred order. The lab picks the first three in each list that the
# workspace actually exposes.
PREFERRED_LLM_ENDPOINTS = [
    "databricks-meta-llama-3-3-70b-instruct",
    "databricks-mixtral-8x7b-instruct",
    "databricks-llama-3-1-8b-instruct",
    "databricks-meta-llama-3-1-70b-instruct",
    "databricks-dbrx-instruct",
]
PREFERRED_EMBEDDING_ENDPOINTS = [
    "databricks-gte-large-en",
    "databricks-bge-large-en",
    "databricks-bge-small-en",
]

# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS for LAB_SCHEMA_FQN via run_sql(),
# then ALTER SCHEMA ... SET TAGS to apply the lab tag.

# YOUR CODE: Create the four Delta tables. Schemas are listed in the markdown
# above. Use CREATE TABLE IF NOT EXISTS, USING DELTA, no LOCATION, no
# partitioning. Apply the lab tag to each table with ALTER TABLE ... SET TAGS.

# YOUR CODE: Insert the five rows from BENCHMARK_PROMPTS into
# BENCHMARK_PROMPTS_TABLE_FQN. The prompts table is a record of what the
# benchmark ran against; the LLM result table references prompt_id.

# Endpoint discovery. Reads every serving endpoint visible to the caller and
# splits them by task. The lab does not write to the result tables here;
# Tasks 2 and 3 do.
def _endpoint_task(ep):
    """Return the task label for a serving endpoint, robust to SDK shape drift."""
    # SDK exposes endpoint.task on most versions; fall back to inspecting the
    # served-entity task if the top-level attribute is None.
    task = getattr(ep, "task", None)
    if task:
        return task
    config = getattr(ep, "config", None)
    served = getattr(config, "served_entities", None) if config else None
    if served:
        for entity in served:
            entity_task = getattr(entity, "task", None)
            if entity_task:
                return entity_task
    return None


all_endpoints = list(w.serving_endpoints.list())
print(f"Discovered {len(all_endpoints)} serving endpoints in workspace.")

chat_endpoint_names = []
embedding_endpoint_names = []
for ep in all_endpoints:
    name = ep.name
    task = _endpoint_task(ep) or ""
    if task in CHAT_TASK_LABELS:
        chat_endpoint_names.append(name)
    elif task in EMBEDDING_TASK_LABELS:
        embedding_endpoint_names.append(name)


def _pick_three(preferred, available):
    """Pick three endpoint names, preferring the documented order."""
    picked = []
    seen = set()
    for name in preferred:
        if name in available and name not in seen:
            picked.append(name)
            seen.add(name)
        if len(picked) == 3:
            return picked
    for name in available:
        if name not in seen:
            picked.append(name)
            seen.add(name)
        if len(picked) == 3:
            return picked
    return picked


LLM_ENDPOINTS = _pick_three(PREFERRED_LLM_ENDPOINTS, chat_endpoint_names)
EMBEDDING_ENDPOINTS = _pick_three(PREFERRED_EMBEDDING_ENDPOINTS, embedding_endpoint_names)

print(f"LLM endpoints for the benchmark ({len(LLM_ENDPOINTS)}):")
for n in LLM_ENDPOINTS:
    print("  -", n)
print(f"Embedding endpoints for the benchmark ({len(EMBEDDING_ENDPOINTS)}):")
for n in EMBEDDING_ENDPOINTS:
    print("  -", n)

if len(LLM_ENDPOINTS) < 3:
    print()
    print("Fewer than three chat endpoints are visible in this workspace. The")
    print("benchmark needs three; configure an external_models endpoint via")
    print("the workspace UI or wait for the workspace to provision the missing")
    print("Databricks-hosted endpoint, then re-run this cell.")
    raise SystemExit(1)
if len(EMBEDDING_ENDPOINTS) < 3:
    print()
    print("Fewer than three embedding endpoints are visible in this workspace.")
    print("Embedding endpoints are Databricks-hosted on Free Edition; check the")
    print("Serving > Endpoints page in the UI and re-run when three are listed.")
    raise SystemExit(1)

print()
print(f"Schema, four tables, prompt seed, and endpoint lists ready for Tasks 2 to 4.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Three distinct LLM endpoints and three distinct embedding
# endpoints recorded. The validator reads from the two result tables, so
# Task 1 alone does not satisfy this checkpoint; Tasks 2 and 3 populate
# the rows. (The brief allows Checkpoint 1 to fire after the benchmark loops
# rather than gating the loops on prior endpoint enumeration.)


def checkpoint_1(session):
    try:
        llm_sql = (
            f"SELECT COUNT(DISTINCT endpoint_name) FROM {LLM_RESULTS_TABLE_FQN}"
        )
        emb_sql = (
            f"SELECT COUNT(DISTINCT endpoint_name) FROM {EMBEDDING_RESULTS_TABLE_FQN}"
        )
        llm_result = run_sql(llm_sql)
        emb_result = run_sql(emb_sql)
        llm_count = int(llm_result["rows"][0][0]) if llm_result["rows"] else 0
        emb_count = int(emb_result["rows"][0][0]) if emb_result["rows"] else 0

        if llm_count < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{LLM_RESULTS_TABLE_FQN} has {llm_count} distinct "
                    f"endpoint_name values; the benchmark must cover at least 3 "
                    f"chat-completion endpoints. Run the Task 2 loop against the "
                    f"three endpoints in LLM_ENDPOINTS."
                ),
            )
        if emb_count < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{EMBEDDING_RESULTS_TABLE_FQN} has {emb_count} distinct "
                    f"endpoint_name values; the benchmark must cover at least 3 "
                    f"embedding endpoints. Run the Task 3 loop against the three "
                    f"endpoints in EMBEDDING_ENDPOINTS."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads two tables and asserts each has at least three distinct endpoint names. If the count is zero, the benchmark loop did not run; if it is one or two, the loop only hit one or two endpoints. Read the error message before peeking further.

</details>

<details><summary>Hint 2 (direction)</summary>

This checkpoint depends on Tasks 2 and 3 producing rows. The lines that need to exist by the time you hit Checkpoint 1 are: (a) CREATE SCHEMA and ALTER SCHEMA SET TAGS, (b) CREATE TABLE x4 for the four Delta tables, (c) ALTER TABLE SET TAGS x4 to apply the lab tag, (d) INSERT INTO benchmark_prompts with the five fixture rows, (e) the Task 2 nested loop that writes 15 rows to `llm_benchmark_results`, (f) the Task 3 nested loop that writes 9 rows to `embedding_benchmark_results`. The constants `LAB_SCHEMA_FQN`, the four `*_TABLE_FQN` strings, `LAB_TAG_KEY`, `LAB_TAG_VALUE`, and `BENCHMARK_PROMPTS` are already in scope.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1 (Tasks 2 and 3 populate the rows the checkpoint reads; their hint blocks show those loops):

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(
    f"ALTER SCHEMA {LAB_SCHEMA_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)

run_sql(
    f"CREATE TABLE IF NOT EXISTS {BENCHMARK_PROMPTS_TABLE_FQN} ("
    f"  prompt_id INT, name STRING, system_message STRING, user_message STRING"
    f") USING DELTA"
)
run_sql(
    f"CREATE TABLE IF NOT EXISTS {LLM_RESULTS_TABLE_FQN} ("
    f"  endpoint_name STRING, endpoint_type STRING, prompt_id INT, "
    f"  latency_ms DOUBLE, prompt_tokens INT, completion_tokens INT, "
    f"  cost_usd_estimate DOUBLE, response_excerpt STRING"
    f") USING DELTA"
)
run_sql(
    f"CREATE TABLE IF NOT EXISTS {EMBEDDING_RESULTS_TABLE_FQN} ("
    f"  endpoint_name STRING, endpoint_type STRING, query_id INT, "
    f"  query_text STRING, embedding_dimension INT, latency_ms DOUBLE, "
    f"  cost_usd_estimate DOUBLE"
    f") USING DELTA"
)
run_sql(
    f"CREATE TABLE IF NOT EXISTS {DECISIONS_TABLE_FQN} ("
    f"  decision_type STRING, chosen_endpoint STRING, "
    f"  constraint_summary STRING, justification_text STRING"
    f") USING DELTA"
)

for fqn in (
    BENCHMARK_PROMPTS_TABLE_FQN,
    LLM_RESULTS_TABLE_FQN,
    EMBEDDING_RESULTS_TABLE_FQN,
    DECISIONS_TABLE_FQN,
):
    run_sql(
        f"ALTER TABLE {fqn} "
        f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
    )

def _escape(s):
    return str(s).replace("'", "''")

values = ", ".join(
    f"({p['id']}, '{_escape(p['name'])}', '{_escape(p['system'])}', '{_escape(p['user'])}')"
    for p in BENCHMARK_PROMPTS
)
run_sql(f"INSERT INTO {BENCHMARK_PROMPTS_TABLE_FQN} VALUES {values}")
```

</details>

**Wallet check.** Still at $0.00 of Foundation Model API spend. One CREATE SCHEMA, four CREATE TABLE, five ALTER TABLE SET TAGS, and one INSERT against the Starter warehouse. The serverless warehouse-seconds budget moves a tick. Tasks 2 and 3 are where the pay-per-token meter starts.

## Task 2: Run the five-prompt LLM benchmark against three chat endpoints

Three endpoints, five prompts each, 15 total chat completions. For every call capture:

- `latency_ms`: wall-clock from the call site to the response, measured with `time.perf_counter_ns()`.
- `prompt_tokens`, `completion_tokens`: from `response.usage`.
- `cost_usd_estimate`: `(prompt_tokens / 1000) * input_per_1k + (completion_tokens / 1000) * output_per_1k`. Rates live in the `PRICING` dict from the setup cell.
- `response_excerpt`: the first 200 characters of `response.choices[0].message.content`, useful for the speed-run QA doc to spot-check.

The lab ships the timing helper. You write the nested loop and the cost math.

**Warm-up call.** First-call cold-start on a Foundation Model API endpoint runs 5 to 20 seconds while the model swaps in. The brief mandates a single warm-up call per endpoint that the benchmark **discards**. The skeleton calls the warm-up function for you; your loop runs the five real prompts after the warm-up.

**INSERT pattern.** The cleanest way to land 15 rows is to build a list of value tuples and write them in one `INSERT INTO ... VALUES (...), (...), ...` statement. String values need single-quote escaping (the helper `_escape` below handles it).

This task burns about 6,000 tokens at a blended pay-per-token rate of roughly 0.30 USD per million tokens. Cost: roughly 0.2 cents. Latency: 60 to 120 seconds end-to-end because of cold-starts.

In [ ]:
# NBVAL_SKIP
# Task 2: nested loop over (LLM_ENDPOINTS x BENCHMARK_PROMPTS) capturing
# latency, token usage, and computed cost. Lab ships time_chat_completion()
# and the warm-up helper; student writes the loop and the cost math, plus
# the bulk INSERT.

def _escape(s):
    return str(s).replace("'", "''")


def time_chat_completion(endpoint_name, system_message, user_message, max_tokens=400):
    """Issue one chat completion against the given endpoint. Returns a dict
    with latency_ms (float), prompt_tokens (int), completion_tokens (int),
    content (str). Raises on transport error so the loop can catch and skip
    cleanly."""
    start = perf_counter_ns()
    response = openai_client.chat.completions.create(
        model=endpoint_name,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
        max_tokens=max_tokens,
        temperature=0.0,
    )
    elapsed_ns = perf_counter_ns() - start
    return {
        "latency_ms": elapsed_ns / 1_000_000.0,
        "prompt_tokens": int(response.usage.prompt_tokens) if response.usage else 0,
        "completion_tokens": int(response.usage.completion_tokens) if response.usage else 0,
        "content": (response.choices[0].message.content or "")[:200] if response.choices else "",
    }


def warm_up_endpoint(endpoint_name):
    """Single ping to wake the endpoint. Result is intentionally discarded."""
    try:
        time_chat_completion(
            endpoint_name,
            "You answer in one word.",
            "Say ready.",
            max_tokens=8,
        )
    except Exception as exc:
        print(f"  warm-up call for {endpoint_name} failed: {exc}")


def cost_for_call(endpoint_name, prompt_tokens, completion_tokens):
    """Look up rates in PRICING. Endpoints not in the dict fall back to the
    70B rate as a conservative upper bound so the cost number is never zero."""
    rates = PRICING.get(endpoint_name) or PRICING.get(
        "databricks-meta-llama-3-3-70b-instruct"
    )
    return (
        (prompt_tokens / 1000.0) * rates["input_per_1k"]
        + (completion_tokens / 1000.0) * rates["output_per_1k"]
    )


print("Warming up the three LLM endpoints. Quirky wait incoming.")
for ep in LLM_ENDPOINTS:
    print(f"  Tapping {ep} on the shoulder so it stops snoozing...")
    warm_up_endpoint(ep)

# YOUR CODE: Build a list called `result_rows` of (endpoint_name, endpoint_type,
# prompt_id, latency_ms, prompt_tokens, completion_tokens, cost_usd_estimate,
# response_excerpt) tuples. For each endpoint in LLM_ENDPOINTS, for each prompt
# in BENCHMARK_PROMPTS, call time_chat_completion(endpoint, prompt["system"],
# prompt["user"]) and cost_for_call(endpoint, prompt_tokens, completion_tokens)
# and append the row. Use "FOUNDATION_MODEL_API" as endpoint_type for every
# Databricks-hosted endpoint.

result_rows = []  # YOUR CODE: populate this list with 15 rows.

# YOUR CODE: After the loop, write the rows in a single INSERT statement.
# Pattern: build a VALUES clause from result_rows and run_sql() once.

print()
print(f"LLM benchmark complete. Rows written to {LLM_RESULTS_TABLE_FQN}: {len(result_rows)}.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: 15 rows in the LLM result table with no nulls on the four
# measured columns and latency_ms > 0 for every row.


def checkpoint_2(session):
    try:
        count_sql = f"SELECT COUNT(*) FROM {LLM_RESULTS_TABLE_FQN}"
        count_result = run_sql(count_sql)
        row_count = int(count_result["rows"][0][0]) if count_result["rows"] else 0
        if row_count != 15:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{LLM_RESULTS_TABLE_FQN} has {row_count} rows; expected "
                    f"exactly 15 (3 endpoints x 5 prompts). Truncate and re-run "
                    f"the Task 2 loop if you over-inserted."
                ),
            )

        null_sql = (
            "SELECT "
            "  SUM(CASE WHEN latency_ms IS NULL THEN 1 ELSE 0 END) AS null_latency, "
            "  SUM(CASE WHEN prompt_tokens IS NULL THEN 1 ELSE 0 END) AS null_prompt, "
            "  SUM(CASE WHEN completion_tokens IS NULL THEN 1 ELSE 0 END) AS null_completion, "
            "  SUM(CASE WHEN cost_usd_estimate IS NULL THEN 1 ELSE 0 END) AS null_cost, "
            "  SUM(CASE WHEN latency_ms <= 0 THEN 1 ELSE 0 END) AS zero_latency "
            f"FROM {LLM_RESULTS_TABLE_FQN}"
        )
        null_result = run_sql(null_sql)
        if not null_result["rows"]:
            return CheckpointResult(
                status="error",
                error_reason="Null-rate query returned no rows.",
            )
        null_latency, null_prompt, null_completion, null_cost, zero_latency = (
            int(v or 0) for v in null_result["rows"][0]
        )
        if null_latency or null_prompt or null_completion or null_cost:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{LLM_RESULTS_TABLE_FQN} has null values: latency_ms={null_latency}, "
                    f"prompt_tokens={null_prompt}, completion_tokens={null_completion}, "
                    f"cost_usd_estimate={null_cost}. Every row must have all four "
                    f"measured columns populated. Check the Task 2 loop is reading "
                    f"response.usage and computing cost via cost_for_call()."
                ),
            )
        if zero_latency:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{LLM_RESULTS_TABLE_FQN} has {zero_latency} rows with "
                    f"latency_ms <= 0. Did you forget to capture perf_counter_ns() "
                    f"around the chat completion call? The lab ships "
                    f"time_chat_completion() which already returns latency_ms; use it."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint either says the row count is wrong (the loop did not run or ran the wrong number of times), or it says one of the four measured columns has nulls (the row was inserted but a value was missing), or it says latency_ms is zero (the timing wrapper was not used). The error message names the failure mode.

</details>

<details><summary>Hint 2 (direction)</summary>

Two nested for-loops. Outer over `LLM_ENDPOINTS`, inner over `BENCHMARK_PROMPTS`. Per iteration, call the shipped `time_chat_completion(endpoint, prompt["system"], prompt["user"])` which already returns latency_ms, prompt_tokens, completion_tokens, and content. Compute cost with the shipped `cost_for_call(endpoint, prompt_tokens, completion_tokens)`. Append a tuple to `result_rows`. After both loops, build one bulk INSERT statement: `INSERT INTO <table> VALUES (..),(..),..` and run it via `run_sql()`. Use the `_escape` helper for any string field that might contain a single quote.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
for endpoint in LLM_ENDPOINTS:
    print(f"Benchmarking {endpoint} across {len(BENCHMARK_PROMPTS)} prompts...")
    for prompt in BENCHMARK_PROMPTS:
        try:
            r = time_chat_completion(
                endpoint,
                prompt["system"],
                prompt["user"],
            )
        except Exception as exc:
            print(f"  prompt {prompt['id']} on {endpoint} raised: {exc}")
            continue
        cost = cost_for_call(endpoint, r["prompt_tokens"], r["completion_tokens"])
        result_rows.append((
            endpoint,
            "FOUNDATION_MODEL_API",
            prompt["id"],
            r["latency_ms"],
            r["prompt_tokens"],
            r["completion_tokens"],
            cost,
            r["content"],
        ))
        print(
            f"  prompt {prompt['id']} ({prompt['name']}): "
            f"{r['latency_ms']:.0f} ms, {r['prompt_tokens']}+"
            f"{r['completion_tokens']} tokens, ${cost:.5f}"
        )

values_clause = ", ".join(
    "("
    f"'{_escape(r[0])}', '{_escape(r[1])}', {r[2]}, {r[3]}, {r[4]}, {r[5]}, "
    f"{r[6]}, '{_escape(r[7])}'"
    ")"
    for r in result_rows
)
run_sql(
    f"INSERT INTO {LLM_RESULTS_TABLE_FQN} ("
    f"  endpoint_name, endpoint_type, prompt_id, latency_ms, prompt_tokens, "
    f"  completion_tokens, cost_usd_estimate, response_excerpt"
    f") VALUES {values_clause}"
)
```

If the loop is interrupted partway through and the table has stale rows, reset before re-running:

```python
run_sql(f"TRUNCATE TABLE {LLM_RESULTS_TABLE_FQN}")
```

</details>

**Wallet check.** Roughly 15 chat completions plus 3 warm-up calls, blended at about 400 tokens per call. Burn so far: about 7,000 tokens, roughly 0.2 cents on Databricks pay-per-token. The wall clock spent more on the warm-up than the actual benchmark. Your morning coffee cost 1,000x more.

## Task 3: Run the three-query embedding benchmark against three embedding endpoints

Three endpoints, three queries each, 9 total embedding calls. For every call capture:

- `latency_ms`: wall-clock from the call site to the response.
- `embedding_dimension`: `len(response.data[0].embedding)`.
- `cost_usd_estimate`: embedding endpoints bill input-only; `(input_tokens / 1000) * input_per_1k` where input_tokens comes from `response.usage.prompt_tokens` if the endpoint reports usage, otherwise `len(query.split())` is the fallback estimate.

Same pattern as Task 2: warm-up call, then the nested loop, then the bulk INSERT. The lab ships `time_embedding(endpoint, text)`.

Expected dimensions per the documented model cards: `databricks-gte-large-en` 1024, `databricks-bge-large-en` 1024, `databricks-bge-small-en` 384. If your workspace exposes a smaller model under a different name, the dimension lookup in Checkpoint 3 is warn-only (the row still passes).

This task burns about 200 tokens total. Cost: a fraction of a cent. Latency: about 5 to 15 seconds end-to-end.

In [ ]:
# NBVAL_SKIP
# Task 3: nested loop over (EMBEDDING_ENDPOINTS x EMBEDDING_QUERIES) capturing
# latency, embedding_dimension, and cost. Lab ships time_embedding(); student
# writes the loop and the bulk INSERT.

def time_embedding(endpoint_name, text):
    """Issue one embedding call against the given endpoint. Returns a dict
    with latency_ms (float), embedding_dimension (int), input_tokens (int).
    The embedding vector itself is read for length only; the benchmark does
    not retain the vectors."""
    start = perf_counter_ns()
    response = openai_client.embeddings.create(
        model=endpoint_name,
        input=text,
    )
    elapsed_ns = perf_counter_ns() - start
    vec = response.data[0].embedding if response.data else []
    input_tokens = (
        int(response.usage.prompt_tokens)
        if response.usage and response.usage.prompt_tokens is not None
        else len(text.split())
    )
    return {
        "latency_ms": elapsed_ns / 1_000_000.0,
        "embedding_dimension": len(vec),
        "input_tokens": input_tokens,
    }


def cost_for_embedding(endpoint_name, input_tokens):
    rates = PRICING.get(endpoint_name) or PRICING.get("databricks-gte-large-en")
    return (input_tokens / 1000.0) * rates["input_per_1k"]


print("Warming up the three embedding endpoints. Polishing the lenses.")
for ep in EMBEDDING_ENDPOINTS:
    print(f"  Asking {ep} to take a deep breath...")
    try:
        time_embedding(ep, "warm up")
    except Exception as exc:
        print(f"  warm-up call for {ep} failed: {exc}")

# YOUR CODE: Build a list called `emb_result_rows` of (endpoint_name,
# endpoint_type, query_id, query_text, embedding_dimension, latency_ms,
# cost_usd_estimate) tuples. For each endpoint in EMBEDDING_ENDPOINTS, for
# each idx, query in enumerate(EMBEDDING_QUERIES, start=1), call
# time_embedding(endpoint, query) and cost_for_embedding(endpoint, input_tokens)
# and append the row. Use "FOUNDATION_MODEL_API" as endpoint_type.

emb_result_rows = []  # YOUR CODE: populate this list with 9 rows.

# YOUR CODE: After the loop, write the rows in a single INSERT statement
# against EMBEDDING_RESULTS_TABLE_FQN.

print()
print(f"Embedding benchmark complete. Rows written to {EMBEDDING_RESULTS_TABLE_FQN}: {len(emb_result_rows)}.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: 9 rows in the embedding result table with no nulls on the
# three measured columns. Embedding dimension is checked against an expected
# map but a mismatch is warn-only (the workspace may have rotated the small
# embedding endpoint name behind a different dimension).


def checkpoint_3(session):
    try:
        count_sql = f"SELECT COUNT(*) FROM {EMBEDDING_RESULTS_TABLE_FQN}"
        count_result = run_sql(count_sql)
        row_count = int(count_result["rows"][0][0]) if count_result["rows"] else 0
        if row_count != 9:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{EMBEDDING_RESULTS_TABLE_FQN} has {row_count} rows; expected "
                    f"exactly 9 (3 endpoints x 3 queries). Truncate and re-run the "
                    f"Task 3 loop if you over-inserted."
                ),
            )

        null_sql = (
            "SELECT "
            "  SUM(CASE WHEN embedding_dimension IS NULL THEN 1 ELSE 0 END) AS null_dim, "
            "  SUM(CASE WHEN latency_ms IS NULL THEN 1 ELSE 0 END) AS null_latency, "
            "  SUM(CASE WHEN cost_usd_estimate IS NULL THEN 1 ELSE 0 END) AS null_cost "
            f"FROM {EMBEDDING_RESULTS_TABLE_FQN}"
        )
        null_result = run_sql(null_sql)
        if not null_result["rows"]:
            return CheckpointResult(
                status="error",
                error_reason="Null-rate query returned no rows.",
            )
        null_dim, null_latency, null_cost = (
            int(v or 0) for v in null_result["rows"][0]
        )
        if null_dim or null_latency or null_cost:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{EMBEDDING_RESULTS_TABLE_FQN} has null values: "
                    f"embedding_dimension={null_dim}, latency_ms={null_latency}, "
                    f"cost_usd_estimate={null_cost}. Every row must have all three "
                    f"columns populated. Check the Task 3 loop is reading "
                    f"len(response.data[0].embedding) and computing cost via "
                    f"cost_for_embedding()."
                ),
            )

        # Dimension lookup is warn-only. A mismatch surfaces in the warnings
        # path but the checkpoint still passes; the brief explicitly allows
        # this because workspace endpoint rotation can change the underlying
        # dimension on a stable endpoint name.
        dim_sql = (
            f"SELECT endpoint_name, MIN(embedding_dimension), MAX(embedding_dimension) "
            f"FROM {EMBEDDING_RESULTS_TABLE_FQN} GROUP BY endpoint_name"
        )
        dim_result = run_sql(dim_sql)
        for row in dim_result["rows"]:
            name = row[0]
            min_dim = int(row[1] or 0)
            max_dim = int(row[2] or 0)
            expected = EXPECTED_EMBEDDING_DIMENSIONS.get(name)
            if expected is not None and (min_dim != expected or max_dim != expected):
                print(
                    f"  warn: {name} returned dimension {min_dim}-{max_dim} but "
                    f"the documented dimension is {expected}. Endpoint may have "
                    f"been rotated; checkpoint still passes."
                )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint says the row count is wrong, or one of the three measured columns has nulls. Embedding dimension mismatches are warn-only and do not fail the checkpoint; if the only issue is a dimension warning, the checkpoint passes. Read the error message before peeking further.

</details>

<details><summary>Hint 2 (direction)</summary>

Two nested loops over `EMBEDDING_ENDPOINTS` and `EMBEDDING_QUERIES`. Use `enumerate(EMBEDDING_QUERIES, start=1)` so `query_id` is 1, 2, 3. The shipped `time_embedding(endpoint, query)` returns latency_ms, embedding_dimension, and input_tokens. Compute cost with `cost_for_embedding(endpoint, input_tokens)`. Build the rows into `emb_result_rows`, then issue one bulk INSERT against `EMBEDDING_RESULTS_TABLE_FQN`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
for endpoint in EMBEDDING_ENDPOINTS:
    print(f"Embedding benchmark on {endpoint}...")
    for query_id, query in enumerate(EMBEDDING_QUERIES, start=1):
        try:
            r = time_embedding(endpoint, query)
        except Exception as exc:
            print(f"  query {query_id} on {endpoint} raised: {exc}")
            continue
        cost = cost_for_embedding(endpoint, r["input_tokens"])
        emb_result_rows.append((
            endpoint,
            "FOUNDATION_MODEL_API",
            query_id,
            query,
            r["embedding_dimension"],
            r["latency_ms"],
            cost,
        ))
        print(
            f"  query {query_id}: dim={r['embedding_dimension']}, "
            f"{r['latency_ms']:.0f} ms, ${cost:.6f}"
        )

values_clause = ", ".join(
    "("
    f"'{_escape(r[0])}', '{_escape(r[1])}', {r[2]}, '{_escape(r[3])}', "
    f"{r[4]}, {r[5]}, {r[6]}"
    ")"
    for r in emb_result_rows
)
run_sql(
    f"INSERT INTO {EMBEDDING_RESULTS_TABLE_FQN} ("
    f"  endpoint_name, endpoint_type, query_id, query_text, "
    f"  embedding_dimension, latency_ms, cost_usd_estimate"
    f") VALUES {values_clause}"
)
```

If the loop is interrupted and you need to reset:

```python
run_sql(f"TRUNCATE TABLE {EMBEDDING_RESULTS_TABLE_FQN}")
```

</details>

**Wallet check.** 9 embedding calls plus 3 warm-up calls. Embedding endpoints bill input-only at fractions of a cent per 1,000 tokens. Burn so far this task: roughly 100 tokens, under 0.001 cents. Running total across Tasks 2 and 3: about 0.2 cents. Still inside the daily quota by four orders of magnitude.

## Task 4: Write two decision rows that pick the LLM and the embedding model and justify each pick

Two INSERT statements against `decisions`. One row with `decision_type = 'llm'`, one with `decision_type = 'embedding'`. Each row carries:

- `chosen_endpoint`: the endpoint name you picked.
- `constraint_summary`: a short string that names the binding constraint (paste from `CONSTRAINT_SCENARIO` or write your own one-liner).
- `justification_text`: one paragraph, **more than 100 characters**, that names the trade-off you considered and references at least one of: `latency`, `cost`, `token`, `recall`, `dimension`, or `context`. Checkpoint 4 enforces both the length and the keyword.

**The constraint scenario.** 100 queries per day. 200 ms p95 latency on first token. 0.50 USD per day total token budget. Source documents chunked at 512 tokens. English only. Requests 50 to 200 tokens, responses 100 to 400 tokens.

**Selection logic to reason through (this is what the exam tests):**

- For the LLM: the 70B endpoint typically misses 200 ms p95 first-token under cold-start and burns the daily budget at 400-token responses. A smaller pay-per-token endpoint (mixtral-class, or whatever your workspace exposes under `databricks-mixtral-8x7b-instruct` or the small-model substitute) fits both latency and cost while delivering sufficient English-only short-chat quality. Match the model to the constraint, not to the leaderboard.
- For the embedding model: chunks max out at 512 tokens, so any model with at least 512 context length is sufficient. Smaller dimensions cost less to store in Vector Search. `databricks-bge-small-en` at 384 dimensions stores roughly 2.7x fewer bytes than the 1024-dim large models with sufficient retrieval recall on short English chunks.

Your justification text is what next quarter's reviewer reads when they revisit this decision. Be specific. Reference the constraint. Name the alternative you considered and why you rejected it.

In [ ]:
# NBVAL_SKIP
# Task 4: write two INSERT statements that persist the picks. Read the
# benchmark results from the two result tables to ground your pick in real
# numbers (the brief framing: "no more we-should-benchmark-first; this is
# the benchmark").

print("LLM benchmark summary (median latency and cost per endpoint):")
llm_summary = run_sql(
    "SELECT endpoint_name, "
    "  ROUND(PERCENTILE(latency_ms, 0.5), 1) AS median_latency_ms, "
    "  ROUND(AVG(cost_usd_estimate), 6) AS avg_cost_per_call "
    f"FROM {LLM_RESULTS_TABLE_FQN} GROUP BY endpoint_name ORDER BY endpoint_name"
)
for row in llm_summary["rows"]:
    print(f"  {row[0]}: median {row[1]} ms, avg ${row[2]} per call")

print()
print("Embedding benchmark summary (median latency and dimension per endpoint):")
emb_summary = run_sql(
    "SELECT endpoint_name, "
    "  ROUND(PERCENTILE(latency_ms, 0.5), 1) AS median_latency_ms, "
    "  MAX(embedding_dimension) AS dim, "
    "  ROUND(AVG(cost_usd_estimate), 8) AS avg_cost_per_call "
    f"FROM {EMBEDDING_RESULTS_TABLE_FQN} GROUP BY endpoint_name ORDER BY endpoint_name"
)
for row in emb_summary["rows"]:
    print(f"  {row[0]}: median {row[1]} ms, dim {row[2]}, avg ${row[3]} per call")

print()
print(f"Constraint scenario: {CONSTRAINT_SCENARIO}")
print()
print("Pick one LLM, pick one embedding model, write the justifications, INSERT both rows.")

# YOUR CODE: Insert a row with decision_type='llm', chosen_endpoint = your pick
# from LLM_ENDPOINTS, constraint_summary referencing the 200 ms p95 and the
# 0.50 USD per day budget, and justification_text > 100 characters that names
# the trade-off and contains at least one of (latency, cost, token, recall,
# dimension, context).

# YOUR CODE: Insert a row with decision_type='embedding', chosen_endpoint =
# your pick from EMBEDDING_ENDPOINTS, constraint_summary referencing the 512-
# token chunk size, and justification_text > 100 characters that names the
# trade-off and contains at least one of the keywords above.

decision_count = run_sql(f"SELECT COUNT(*) FROM {DECISIONS_TABLE_FQN}")
count_now = int(decision_count["rows"][0][0]) if decision_count["rows"] else 0
print()
print(f"Decisions table now contains {count_now} rows (expected 2 for Checkpoint 4).")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: exactly 2 rows in the decisions table, one per decision_type,
# each with a non-null chosen_endpoint and a justification_text > 100 chars
# that contains at least one constraint keyword.


def checkpoint_4(session):
    try:
        sql = (
            "SELECT decision_type, chosen_endpoint, justification_text "
            f"FROM {DECISIONS_TABLE_FQN} ORDER BY decision_type"
        )
        result = run_sql(sql)
        rows = result["rows"]
        if len(rows) != 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{DECISIONS_TABLE_FQN} has {len(rows)} rows; expected exactly "
                    f"2 (one decision_type='llm', one decision_type='embedding'). "
                    f"Truncate and re-INSERT if you over-inserted."
                ),
            )

        types_seen = set()
        keyword_set = {"latency", "cost", "token", "recall", "dimension", "context"}
        for decision_type, chosen_endpoint, justification_text in rows:
            dtype = (decision_type or "").lower()
            types_seen.add(dtype)
            if dtype not in ("llm", "embedding"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"decision_type={decision_type!r} is not one of 'llm' or "
                        f"'embedding'. Insert exactly one row per type."
                    ),
                )
            if not chosen_endpoint or not str(chosen_endpoint).strip():
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"decision_type={dtype!r} has a null or empty chosen_endpoint. "
                        f"Pick an endpoint name from the benchmark results."
                    ),
                )
            jt = justification_text or ""
            if len(jt) <= 100:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"decision_type={dtype!r} has justification_text of length "
                        f"{len(jt)}; must be > 100 characters. Reviewers next quarter "
                        f"need the reasoning, not a one-liner."
                    ),
                )
            lower_jt = jt.lower()
            if not any(kw in lower_jt for kw in keyword_set):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"decision_type={dtype!r} justification_text is missing every "
                        f"constraint keyword. Reference at least one of: latency, "
                        f"cost, token, recall, dimension, context."
                    ),
                )

        if types_seen != {"llm", "embedding"}:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"decision_types present: {sorted(types_seen)}. Expected exactly "
                    f"{{'llm', 'embedding'}}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint fails for one of four reasons: row count is not 2, decision_type is not one of 'llm' or 'embedding', chosen_endpoint is empty, justification_text is too short or missing a constraint keyword. Read the error message before peeking.

</details>

<details><summary>Hint 2 (direction)</summary>

Two INSERT INTO statements against `decisions`. The keyword list (`latency`, `cost`, `token`, `recall`, `dimension`, `context`) is case-insensitive, but every justification still needs at least one. The "more than 100 characters" rule applies separately to each row, not to the two combined. Pick the smaller chat endpoint for the LLM row (the constraint scenario is exactly the "small model fits, 70B does not" pattern from Question 2 in the exam-style practice). Pick the smaller embedding endpoint for the embedding row when the constraint priority is "cost and latency over quality" on 512-token chunks (Question 1).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4 (pick endpoint names that match what your workspace actually exposes; the names below assume the documented Free Edition endpoints):

```python
llm_pick = "databricks-mixtral-8x7b-instruct"  # or LLM_ENDPOINTS[1], whichever small-model the workspace has
emb_pick = "databricks-bge-small-en"           # smallest embedding for the 512-token cost/latency scenario

llm_constraint = (
    "100 queries per day, 200 ms p95 first-token latency, 0.50 USD per day token budget, English only."
)
emb_constraint = (
    "Source chunks at 512 tokens, prefer cost and latency over absolute retrieval recall."
)

llm_justification = (
    f"Chose {llm_pick} because the smaller pay-per-token endpoint hit the 200 ms p95 latency target "
    f"under cold-start and fits the 0.50 USD per day token budget at 100 queries with 400-token "
    f"responses, while the 70B endpoint missed the latency target and would have burned the daily "
    f"budget. English-only short chat does not justify the leaderboard model."
)
emb_justification = (
    f"Chose {emb_pick} because chunks max out at 512 tokens, so a 512-context 384-dimension embedding "
    f"is sufficient. Smaller dimension stores roughly 2.7x fewer bytes in Vector Search than the 1024-dim "
    f"large models with acceptable recall on short English chunks. Cost and latency dominated the decision."
)

run_sql(
    f"INSERT INTO {DECISIONS_TABLE_FQN} VALUES ("
    f"  'llm', '{llm_pick}', '{_escape(llm_constraint)}', '{_escape(llm_justification)}'"
    f")"
)
run_sql(
    f"INSERT INTO {DECISIONS_TABLE_FQN} VALUES ("
    f"  'embedding', '{emb_pick}', '{_escape(emb_constraint)}', '{_escape(emb_justification)}'"
    f")"
)
```

If you over-insert and the row count goes above 2:

```python
run_sql(f"TRUNCATE TABLE {DECISIONS_TABLE_FQN}")
```

</details>

**Wallet check.** Two more SQL statements against the warehouse. No Foundation Model API calls in this task. Session total so far: roughly 0.2 cents on pay-per-token. The benchmark itself spent less than a single second of a Starbucks pour.

## Cleanup

Time to tear it all down. The cell below runs through your manifest in reverse-creation order (decisions, embedding results, LLM results, prompts, then the schema with `CASCADE`), then double-checks UC information_schema to confirm everything is gone. Re-running is safe.

No hourly-billed resources to worry about. The pay-per-token spend already settled at request time; nothing in Free Edition charges by the hour for this lab. The four Delta tables and the schema are the only persistent artifacts and they all get dropped here.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter. No critical (hourly-billed)
# resources in this lab, so the canonical summary always reports zero
# critical destructions.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.002.** Roughly 18 Foundation Model API calls (15 chat completions plus 3 chat warm-ups) at a blended 0.30 USD per million tokens plus 12 embedding calls (9 plus 3 warm-ups) at fractions of a cent. Databricks Free Edition swallowed the compute and the metastore. If you wired up an external_models endpoint via OpenAI, add 1 to 5 cents on the OpenAI side. The Delta tables and the schema were destroyed by the cleanup cell above, so the daily compute quota is the only thing this session touched and it resets at 00:00 UTC. Your morning coffee cost more than this lab.

## Reflection

These are not graded. They are for you.

1. Walk through the trade-offs you considered when picking the LLM. Name the constraint that drove the pick (latency, cost, license, context length, multilingual support) and the constraint you were willing to compromise on. What would you have picked if the daily query volume were 10,000 instead of 100, and why does the answer change with volume?

2. The embedding model dimension matters for both retrieval quality and Vector Search storage cost. A 1024-dim embedding stores roughly 4x the bytes of a 256-dim embedding for the same number of chunks. When would the storage-cost trade-off matter more than the retrieval-quality trade-off, and how would you measure quality on your specific corpus before locking in the smaller model?

## Exam-Style Practice

**Question 1 (Domain 3, embedding context length per the official sample-question pattern):**

A GenAI engineer is building a RAG application whose source documents have been chunked to a maximum of 512 tokens per chunk. Cost and latency are more important than quality. Which embedding model context length is the best fit?

A. context length 512, smallest model 0.13 GB, embedding dimension 384.
B. context length 514, smallest model 0.44 GB, embedding dimension 768.
C. context length 2048, smallest model 11 GB, embedding dimension 2560.
D. context length 32768, smallest model 14 GB, embedding dimension 4096.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: the chunks max out at 512 tokens, so a 512-context embedding model is sufficient. The smallest model size (0.13 GB) and the smallest embedding dimension (384) directly minimize cost and latency. Picking a longer-context model would over-provision compute and storage without quality return.
- B is wrong: a 514-context model adds nothing for 512-token chunks, and the larger size adds latency without improving quality.
- C and D are wrong: longer context lengths and higher dimensions raise both cost and latency without improving quality on 512-token chunks. The exam tests the "match the context length to the chunk size" decision (Question 4 in the official sample-question set is this exact pattern).

</details>

**Question 2 (Domain 1, LLM selection under a cost+latency constraint):**

A GenAI engineer is choosing an LLM for an internal chat assistant with these constraints: 100 queries per day, 200 ms p95 latency on first token, 0.50 USD per day total token budget, English-only, requests are 50 to 200 tokens, responses are 100 to 400 tokens. Which Foundation Model API endpoint is the best fit?

A. `databricks-meta-llama-3-3-70b-instruct` (70B params, multilingual, ~600 ms p95 first token under typical load).
B. `databricks-mixtral-8x7b-instruct` or the equivalent smaller-model endpoint (~150 ms p95 first token, sufficient quality for short English-only chat at much lower cost-per-1k-tokens).
C. An external `gpt-5-turbo` endpoint via external_models, because it is the best model.
D. `databricks-meta-llama-3-3-405b-instruct` (provisioned throughput only, lowest latency per token at scale).

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: 70B on pay-per-token typically misses the 200 ms p95 first-token target under cold-start and would exhaust the 0.50 USD daily budget at 100 queries with 400-token responses.
- B is correct: a smaller pay-per-token endpoint (mixtral-class or the current small-model substitute) hits the latency target, fits the budget, and provides sufficient quality for English-only short chat. Match the model to the constraint, not to the leaderboard.
- C is wrong: the best-on-benchmark model is rarely the right pick under a tight latency+cost constraint. The exam tests constraint-driven selection over leaderboard-driven selection.
- D is wrong: 405B is provisioned-throughput-only (not pay-per-token on Free Edition) and is over-provisioned for 100 queries per day; the per-hour bill would blow the daily budget.

</details>